In [ ]:
import uproot
import awkward as ak

import pandas as pd

import matplotlib.pylab as plt
import numpy as np

import hist
from hist import Hist

import time

import myPIDselector

import math

import os

data = ak.from_parquet("Background_and_signal_SP_modes_All_runs.parquet")
blindD = ak.from_parquet("Data_All_runs_BLINDED.parquet")

B_D1_IDX_FIELD            = 'Bd1Idx'
B_D2_IDX_FIELD            = 'Bd2Idx'

# Lambda daughter fields
LAMBDA_PROTON_IDX_FIELD   = 'Lambda0d1Idx'
LAMBDA_PROTON_LUND_FIELD  = 'Lambda0d1Lund'

 
PROTON_TRKIDX_FIELD       = 'pTrkIdx'
PROTON_SELECTORS_FIELD    = 'pSelectorsMap'
TRK_LUND_FIELD            = 'TRKLund'

MES_FIELD                 = 'BpostFitMes'
DELTAE_FIELD              = 'BpostFitDeltaE'

SPMODE_FIELD              = 'spmode'

plt.close()

def calculate_bits_for_PID_selector(trkidx, trk_selector_map, verbose=0):
    
    bits = None

    # If there is no trk index passed in, just calculate the bits for
    # all of the tracks
    if trkidx is None:
        trkidx = ak.local_index(trk_selector_map)

    # Grab the tracks that map on to the particle/collection we are interested in 
    subset_of_trk_selector_map = trk_selector_map[trkidx]
    if verbose:
        print("values in the subset of the trk selector map")
        print(subset_of_trk_selector_map)
        print()
        
    # We need this to convert the numbers in the selector to binary
    binary_repr_vec = np.vectorize(np.binary_repr)

    # Grab the number of entries in each so we can unflatten this later
    counts = ak.num(subset_of_trk_selector_map)
    
    # Now get the binary representation (as a string) for the flattened subset
    binrep = binary_repr_vec(ak.flatten(subset_of_trk_selector_map), width=32)

    if verbose:
        print("binary representation of selector map")
        print(binrep)
        print()

    # Convert the string to integers
    tempbits = np.array(binrep).astype(int)
    bits = ak.unflatten(tempbits,counts)

    if verbose:
        print("flattened integer representation of selectors as binary (int)")
        print(tempbits)
        print()
        print("unflattened integer representation of selectors as binary (int)")
        print(bits)
        print()

    return bits
def mask_PID_selection(bits, selector, pid_map_object):


    bit_to_look_for = pid_map_object.selectors.index(selector)

    place = int(math.pow(10,bit_to_look_for))
    #print(place)

    mask = bits // place % 10

    mask_bool = ak.values_astype(mask,bool)

    return mask_bool
def get_info_for_PID_masks(ak_arr, verbosity=0):

    # Proton and pion information from the Lambda decay
    # These are the index of the proton (d1) and pion (d2) in those lists
    L1d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd1Idx]
    L1d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd1Idx]

    L2d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd2Idx]
    L2d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd2Idx]


    
     
    #d1lund = ak_arr['Lambda0d1Lund']
    #d2lund = ak_arr['Lambda0d2Lund']
    
    #Bd2idx = ak_arr['Bd2Idx']
    #Bd2lund = ak_arr['Bd2Lund']
    
    trkidx_proton = ak_arr['pTrkIdx']
    trk_selector_map_proton = ak_arr['pSelectorsMap']
    
    trkidx_pion = ak_arr['piTrkIdx']
    trk_selector_map_pion = ak_arr['piSelectorsMap']

    indices_and_maps = {}
    indices_and_maps['L1d1idx'] = L1d1idx
    indices_and_maps['L1d2idx'] = L1d2idx
    indices_and_maps['L2d1idx'] = L2d1idx
    indices_and_maps['L2d2idx'] = L2d2idx
    #indices_and_maps['Bd2idx'] = Bd2idx

    indices_and_maps['trkidx_proton'] = trkidx_proton
    indices_and_maps['trk_selector_map_proton'] = trk_selector_map_proton

    indices_and_maps['trkidx_pion'] = trkidx_pion
    indices_and_maps['trk_selector_map_pion'] = trk_selector_map_pion

    return indices_and_maps
def PID_masks(indices_and_maps, \
              lam1p_selector,\
              lam1pi_selector, \
              lam2p_selector,\
              lam2pi_selector, \
             verbosity=0):
                #change Bpselector
    L1d1idx = indices_and_maps['L1d1idx']
    L1d2idx = indices_and_maps['L1d2idx']
    L2d1idx = indices_and_maps['L2d1idx']
    L2d2idx = indices_and_maps['L2d2idx']
    #Bd2idx = indices_and_maps['Bd2idx']

    trkidx_proton = indices_and_maps['trkidx_proton']
    trk_selector_map_proton = indices_and_maps['trk_selector_map_proton']

    trkidx_pion = indices_and_maps['trkidx_pion']
    trk_selector_map_pion = indices_and_maps['trk_selector_map_pion']
    
    # Proton
    pbits = calculate_bits_for_PID_selector(trkidx_proton, trk_selector_map_proton, verbose=verbosity)
    # Pion
    pibits = calculate_bits_for_PID_selector(trkidx_pion, trk_selector_map_pion, verbose=verbosity)
    
    
    mask_bool_protonL1 = mask_PID_selection(pbits[L1d1idx], lam1p_selector, pps)
        
    mask_bool_pionL1 = mask_PID_selection(pibits[L1d2idx], lam1pi_selector, pips)

    mask_bool_protonL2 = mask_PID_selection(pbits[L2d1idx], lam2p_selector, pps)
        
    mask_bool_pionL2 = mask_PID_selection(pibits[L2d2idx], lam2pi_selector, pips)

    return mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2
def indices_to_booleans(indices, array_to_slice):
    """Convert ragged integer indices to a boolean mask over array_to_slice."""
    whole_set, in_set = ak.unzip(ak.cartesian(
        [ak.local_index(array_to_slice), indices], nested=True
    ))
    return ak.any(whole_set == in_set, axis=-1)
def ragged_element_pick(ragged_arr, flat_idx):
    """
    For each event i, return ragged_arr[i][flat_idx[i]].

    Parameters
    ----------
    ragged_arr : ak.Array, shape [n_events, var]
    flat_idx   : array-like, shape [n_events]  (integer index per event)

    Returns
    -------
    np.ndarray, shape [n_events]
    """
    flat_content = ak.flatten(ragged_arr)
    counts       = ak.num(ragged_arr)

    offsets = np.zeros(len(counts) + 1, dtype=np.int64)
    np.cumsum(ak.to_numpy(counts), out=offsets[1:])

    idx_np = ak.to_numpy(flat_idx).astype(np.int64)
    global_indices = offsets[:-1] + idx_np

    return ak.to_numpy(flat_content)[global_indices]
def build_antilambda_antimask(ak_arr, pps,
                               selector,
                               verbose=0):
    """
    Antibaryon veto for B0 -> Lambda Lambda (BNV).

    REQUIRES: single-B-candidate cut already applied (exactly one B per event).

    Returns
    -------
    charge_test : awkward array of booleans
        Per-track flag — True where a rest-of-event track is an antiproton.
    """

    n = len(ak_arr)

     # 1.  Squeeze B-candidate dimension
    #bd1idx = ak.to_numpy(ak_arr[B_D1_IDX_FIELD][:, 0])   # scalar per event
    bd1idx = ak.to_numpy(ak.firsts(ak_arr[B_D1_IDX_FIELD], axis=1))

    #bd2idx = ak.to_numpy(ak_arr[B_D2_IDX_FIELD][:, 0])
    bd2idx = ak.to_numpy(ak.firsts(ak_arr[B_D2_IDX_FIELD], axis=1))

    if verbose:
        print(f"bd1idx  {bd1idx[:5]}")
        print(f"bd2idx  {bd2idx[:5]}")
        print()

     
    # 2.  Chain: B daughter Lambda index -> that Lambda's proton index
     
    all_lam_proton_idx  = ak_arr[LAMBDA_PROTON_IDX_FIELD]
    all_lam_proton_lund = ak_arr[LAMBDA_PROTON_LUND_FIELD]

    # Proton-list index for each B-daughter Lambda (flat numpy, one per event)
    lam1_pidx = ragged_element_pick(all_lam_proton_idx, bd1idx)
    lam2_pidx = ragged_element_pick(all_lam_proton_idx, bd2idx)

    if verbose:
        print(f"lam1_pidx (proton-list index from Λ₁)  {lam1_pidx[:5]}")
        print(f"lam2_pidx (proton-list index from Λ₂)  {lam2_pidx[:5]}")
        print()

     #     qlam1 = +1 for Λ → p π⁻,  -1 for Λ̄ → p̄ π⁺
    lam1_lund = ragged_element_pick(all_lam_proton_lund, bd1idx)
    lam2_lund = ragged_element_pick(all_lam_proton_lund, bd2idx)

    qlam1 = np.sign(lam1_lund).astype(float)   # shapes
    qlam2 = np.sign(lam2_lund).astype(float)

    if verbose:
        print(f"qlam1 (charge of Λ₁ proton)  {qlam1[:5]}")
        print(f"qlam2 (charge of Λ₂ proton)  {qlam2[:5]}")
        print()

    qtrk = ak_arr[TRK_LUND_FIELD] / np.abs(ak_arr[TRK_LUND_FIELD])

     # 4.  Map proton-list indices to track-list indices
    trk_selector_map = ak_arr[PROTON_SELECTORS_FIELD]
    trkidx_proton    = ak_arr[PROTON_TRKIDX_FIELD]

    # Pick the track index for each signal proton (flat numpy)
    lam1_trkidx_flat = ragged_element_pick(trkidx_proton, lam1_pidx)
    lam2_trkidx_flat = ragged_element_pick(trkidx_proton, lam2_pidx)

    # Wrap as length-1 ragged for indices_to_booleans
    lam1_trkidx = ak.unflatten(lam1_trkidx_flat, 1)
    lam2_trkidx = ak.unflatten(lam2_trkidx_flat, 1)

    if verbose:
        print(f"lam1_trkidx  {lam1_trkidx[:5]}")
        print(f"lam2_trkidx  {lam2_trkidx[:5]}")
        print()

    bool_idx1 = indices_to_booleans(lam1_trkidx, trk_selector_map)
    bool_idx2 = indices_to_booleans(lam2_trkidx, trk_selector_map)

    if verbose:
        print(f"bool_idx1 (Λ₁ proton)  {bool_idx1[:3]}")
        print(f"bool_idx2 (Λ₂ proton)  {bool_idx2[:3]}")
        print()

    
    # 5.  PID selection on ALL proton candidates  
    pbits     = calculate_bits_for_PID_selector(None, trk_selector_map)
    mask_bool = mask_PID_selection(pbits, selector, pps)

    if verbose:
        print(f"Tracks passing {selector}:  {ak.sum(ak.num(pbits[mask_bool]))}")
        print()

     # 6.  Charge test on REST-OF-EVENT proton candidates
     
    other_mask  = ~(bool_idx1 | bool_idx2) & mask_bool
    charge_test = qtrk[other_mask] == -qlam1

    if verbose:
        print(f"charge_test  {charge_test[:3]}")

   
    have_opp       = ak.num(charge_test[charge_test])
    nhave_opp      = len(have_opp[have_opp > 0])
    ndont_have_opp = len(have_opp[have_opp == 0])

    return charge_test
def apply_antibaryon_veto(ak_arr, pps, selector):
    """Return a per-event boolean: True = passes veto (no antiproton found)."""
    ct       = build_antilambda_antimask(ak_arr, pps, selector=selector, verbose=0)
    have_opp = ak.num(ct[ct])
    return have_opp == 0

pps = myPIDselector.PIDselector("p")
pips = myPIDselector.PIDselector("pi")




d1b = data[(data['nB']==1)]
lamidx = d1b['Lambda0MCIdx']
mcLund = d1b['mcLund']
lamd1idx = d1b['Lambda0d1Idx']
lamd2idx = d1b['Lambda0d2Idx']
pridx = d1b["pMCIdx"]
piidx = d1b["piMCIdx"]
mothIdx = d1b['mothIdx']
dauLen = d1b['dauLen']
pti = d1b['pTrkIdx']
tmi = d1b["TRKMCIdx"]
mct = d1b['mccosth']
nT = d1b['nTracks']



pdg_to_name = {21: "g", 22: "gamma",    23: "Z0",    24: "W+",    -24: "W-",    25: "H0",    11: "e-",     -11: "e+",\
12: "nu_e",    -12: "anti_nu_e",    13: "mu-",    -13: "mu+",    14: "nu_mu",    -14: "anti_nu_mu",15: "tau-",    -15: "tau+",    16: "nu_tau",    -16: "anti_nu_tau",    1: "d", -1: "anti_d",\
2: "u", -2: "anti_u",    3: "s", -3: "anti_s",    4: "c", -4: "anti_c",    5: "b", -5: "anti_b",    6: "t", -6: "anti_t",\
111: "pi0",    211: "pi+",    -211: "pi-",213:"rho+",-213:"anti rho+",    221: "eta",    321: "K+",    -321: "K-",    311: "K0",    -311: "anti_K0",\
130: "K_L0",    310: "K_S0",    411: "D+", -411: "D-", 413:"D*+", -413:"anti D*+", 433:"D*s+", -433:"anti D*s+",    421: "D0", -421: "anti_D0",    431: "D_s+", -431: "D_s-",\
511: "B0", -511: "anti_B0",    521: "B+", -521: "B-",    531: "B_s0", -531: "anti_B_s0",    2212: "p+",    -2212: "anti_p",\
2112: "n",    -2112: "anti_n",    3122: "Lambda0",    -3122: "anti_Lambda0",    3222: "Sigma+",-3222:"AntiSigma+",   3112: "Sigma-",    3212: "Sigma0",-3212:"Sigma0",\
3322: "Xi0",    3312: "Xi-",    3334: "Omega-", 10022: "10022", 2212: "p",    2112: "n",    2224: "Delta++",    2214: "Delta+",\
2114: "Delta0",    1114: "Delta-",4122: "Lambda_c+",    4222: "Sigma_c++",    4212: "Sigma_c+",    4112: "Sigma_c0",    4224: "Sigma*_c++",\
4214: "Sigma*_c+",    4114: "Sigma*_c0", -4114: "anti Sigma*_c0",    4232: "Xi_c+",    4132: "Xi_c0",    4322: "Xi'_c+",    4312: "Xi'_c0",    4324: "Xi*_c+",    4314: "Xi*_c0",\
4332: "Omega_c0",    4334: "Omega*_c0",    4412: "Xi_cc+",    4422: "Xi_cc++", -3224: "Sigma_*-", 3224: "Sigma_*+", -313: "anti K*0", 313: "K*0", 3314: "xi*-",\
    -3314: "antiXi*-", -3312: "antiXi-", -3322: "antiXi0",-2224: "antiDelta++", 70553: '70553', 10113:"b10meson",-10113:"anti b10meson",}

"""
idx1237List = []
d1b1237Mask = (d1b['spmode']=="1237")

for i in range(len(d1b)):
    if d1b1237Mask[i]:
        idx1237List+=[i]

for idx in range(2):
    i = idx1237List[idx]
    print(mcLund[idx1237List[idx]])
    for j in range(len(mcLund[idx1237List[idx]])):
        particle = mcLund[idx1237List[idx]][j]
        print(f"{pdg_to_name[particle]:<12}/{particle:5.0f}/{pdg_to_name[mcLund[i][mothIdx[i][j]]]:<12}/{mcLund[i][mothIdx[i][j]]:5.0f}/{dauLen[i][j]}")
"""




idx998List = [721, 1047, 1352, 2194, 2299, 2588, 4641, 4908, 5406, 5826, 6270, 6817, 6949, 7090, 7254, 7436, 
              8414, 9412, 9451, 10023, 10062, 10333, 10619, 10695, 11114, 13092, 13113, 13407, 13439, 13604, 
              13672, 13676, 13776, 13906, 14186, 14367, 14424, 14573, 15034, 15054, 15386, 15393, 15429, 15929, 
              16244, 16287, 16301, 16572, 17077, 17194, 17816, 18458, 18536, 18862, 19436, 19937, 20158, 20287, 
              20769, 20794, 21069, 21073, 21712, 21949, 22070, 22554, 23090, 23199, 23340, 23371, 23985, 24017, 
              24494, 24569, 24757, 24873, 25354, 25396, 25527, 25574, 25670, 25773, 26167, 26593, 27323, 27514, 
              27619, 27776, 27938, 28393, 28670, 28890, 29299]

idx1005List = [32221, 33158, 35495, 38459, 41081, 41520, 41826, 42239, 49405, 50306, 53846, 55276]  
trkL998 = [] 
trkL1005 = []

for i in idx998List:
    #print(f"Event {i}, First lambda lund: {mcLund[i][lamidx[i]][0]}, d1 Lund: {mcLund[i][pridx[i][lamd1idx[0]]][0]}, d2 Lund: {mcLund[i][piidx[i][lamd2idx[0]]][0]}")
    #print(f"Event {i}, Second lambda lund: {mcLund[i][lamidx[i]][1]}, d1 Lund: {mcLund[i][pridx[i][lamd1idx[1]]][1]}, d2 Lund: {mcLund[i][piidx[i][lamd2idx[1]]][1]}")
    #for j in range(len(mcLund[i])):
    #particle = mcLund[i][j]
    
    #antiPList = []
    #antiPC = 0
    #PList = []
    #PC = 0
    #print('trk lund:', d1b['TRKLund'][i])
    #print("Index of tracks of protons:",pti[i])
    #print("Index of protons in MC:",pridx[i])
    #print("Index in MC of tracks:",tmi[i])

    #trkL998 += [nT[i]]

    for j in range(len(mcLund[i])):
        particle = mcLund[i][j]
        print(f"Name: {pdg_to_name[particle]:<12} /   PID: {particle:5.0f}    /  Mother name: {pdg_to_name[mcLund[i][mothIdx[i][j]]]:<12} /  Mother PID: {mcLund[i][mothIdx[i][j]]:5.0f} /   Num daughters: {dauLen[i][j]}")
        

for i in idx1005List:
    trkL1005 += [nT[i]]


    """
    for j in range(len(mcLund[i])):
        if mcLund[i][j]==-2212:
            antiPList+=[mct[i][j]]
            antiPC+=1
        elif mcLund[i][j]==[2212]:
            PList+=[mct[i][j]]
            PC+=1
    

    idxDict[i] = [antiPList,antiPC,PList,PC]
    """
    """print('trk lund:', d1b['TRKLund'][721])
        print("Index of tracks of protons:",pti[i])
        print("Index of protons in MC:",pridx[i])
        print("Index in MC of tracks:",tmi[i])

        for val in tmi[i]:
            print(mcLund[i][val])
        print("MC val:", mcLund[721][15],mcLund[721][22])
        print("mccosth for first proton:",mct[i][15],"   mccosth for second proton:", mct[i][22])
        trk_selector_map_proton = d1b['pSelectorsMap'][i]
        #print(trk_selector_map_proton)
        #pbits = calculate_bits_for_PID_selector(pti[i], trk_selector_map_proton)
        binary_repr_vec = np.vectorize(np.binary_repr)
        binrep = binary_repr_vec(trk_selector_map_proton[pti[i]], width=32)
        print("Binary pass/fail of protons:",binrep)"""
  

        #print(f"{pdg_to_name[particle]:<12}/{particle:5.0f}/{pdg_to_name[mcLund[i][mothIdx[i][j]]]:<12}/{mcLund[i][mothIdx[i][j]]:5.0f}/{dauLen[i][j]}")
    #print()
"""
for key,value in idxDict.items():
    print(f"Index: {key}, count ~p: {value[1]}, count p: {value[3]}")
    print(value[0])
    print(value[2])
"""
trkLS = []
pidChoiceDict = {}
for cutI in [0]:
    for pidChoice in [157]:# 157, 202]:#,1 112, 256, 121]:
        if pidChoice == 1:
            myPMask = "SuperLooseKMProtonSelection"
            myPShort = "SL Pro"
            myPiMask = "SuperLooseKMPionMicroSelection"
            myPiShort = 'SL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == 157:
            myPMask = "VeryLooseKMProtonSelection"
            myPShort = "VL Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == 202:
            myPMask = "VeryLooseKMProtonSelection"
            myPShort = "VL Pro"
            myPiMask = "TightKMPionMicroSelection"
            myPiShort = 'T   Pi'
            pidChoiceDict[pidChoice] =(myPShort,myPiShort)
        elif pidChoice == 112:
            myPMask = "VeryLooseKMProtonSelection"
            myPShort = "VL Pro"
            myPiMask = "LooseKMPionMicroSelection"
            myPiShort = 'L   Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == 256:
            myPMask = "LooseKMProtonSelection"
            myPShort = "L   Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == 121:
            myPMask = "SuperLooseKMProtonSelection"
            myPShort = "SL Pro"
            myPiMask = "SuperTightKMPionMicroSelection"
            myPiShort = 'ST Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == -1:
            myPMask = "TightKMProtonSelection"
            myPShort = "T   Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == -2:
            myPMask = "VeryTightKMProtonSelection"
            myPShort = "VT Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == -3:
            myPMask = "SuperTightKMProtonSelection"
            myPShort = "ST Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == 0:
            myPMask =  "SuperLooseKMProtonSelection"
            myPiMask = "SuperLooseKMPionMicroSelection"
            pidChoiceDict[pidChoice] = ("None","None")

        numBCan = data['nB']
        bCanMask = (numBCan==1)

        d1b = data[bCanMask]
        

        indices_and_maps = get_info_for_PID_masks(d1b, verbosity=0)

        mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2 = PID_masks(indices_and_maps, \
                    lam1p_selector=myPMask, \
                    lam1pi_selector=myPiMask, \
                    lam2p_selector=myPMask, \
                    lam2pi_selector=myPiMask, \
                    verbosity=0)
    
        
        maxSigCut,minSigCut = (10,10)
            
        bkgE=0
        dataB = False
        

        mes = d1b['BpostFitMes']
        de = d1b['BpostFitDeltaE']

        meslo = 5.2
        mesloS = 5.27
        meshiS = 5.3
        DeltaElo = -0.2
        DeltaEhi = 0.2
        DeltaEloS = -0.07
        DeltaEhiS = 0.07

        mask_pid = mask_bool_pionL1 & mask_bool_protonL1 & mask_bool_pionL2 & mask_bool_protonL2

        preSel_fit = (mes>meslo)&(de<DeltaEhi)&(de>DeltaElo)

        mask_sig = (mes>=mesloS) & (mes<=meshiS) & (de>=DeltaEloS) & (de<=DeltaEhiS)

        mask_sideband = (((mes<mesloS) | (mes>meshiS)) | ((de<DeltaEloS) | (de>DeltaEhiS)))

        firstMask = mask_pid&preSel_fit

        lamconflsig = d1b['Lambda0postFitFlightSignificance']
        lamuncmass = d1b['Lambda0_unc_Mass']
        


        lammass_world_average = 1.115683
        width = 0.003
        lo = lammass_world_average - width
        hi = lammass_world_average + width
            

        mask_fl = (lamconflsig>=minSigCut)
        mask_upper_fl = (lamconflsig>=maxSigCut)

        flightSigMask = (ak.all(mask_fl,axis=1))&(ak.any(mask_upper_fl, axis=1))



        lamuncmassMask = (lamuncmass[:,0]>=lo)&(lamuncmass[:,0]<=hi)& (lamuncmass[:,1]>=lo)&(lamuncmass[:,1]<=hi)
        

        vetoMask = apply_antibaryon_veto(d1b, pps, myPMask)

        finalMask = flightSigMask&lamuncmassMask&preSel_fit&mask_pid&vetoMask

        in_signal = preSel_fit&mask_sig
        in_sideband = preSel_fit&mask_sideband

        scaleD = {"998": 424.3/1719, "1005":424.3/868.3, "-999":1}
        denomD = {'998': 29798, '1005':28500,'-999':44755}
        effDict = {}
        n_srD = {}
        n_sbD = {}

        
        idxL998 = []
        idxL1005 = []
        idxL999 = []
        idxL1005CUT = []
        idxL998CUT = []
        for sp in ['998','1005','-999']:
            
            spmask = (d1b['spmode']==sp)
            
            cut_mask = finalMask&spmask

            effDict[sp]=ak.sum(cut_mask)/(ak.sum(data['spmode']==sp))
            n_srD[sp]=ak.sum(cut_mask&in_signal)*scaleD[sp]
            n_sbD[sp]=ak.sum(cut_mask&in_sideband)*scaleD[sp]

            cut_masknp = ak.to_numpy(cut_mask&spmask)
            if sp == '998':
                for i in range(len(cut_masknp)):
                    if cut_masknp[i]:
                        idxL998 += [i]
                    elif spmask[i]:
                        idxL998CUT += [i]
            if sp == '1005':
                for i in range(len(cut_masknp)):
                    if cut_masknp[i]:
                        idxL1005 += [i]
                    elif spmask[i]:
                        idxL1005CUT += [i]

            elif sp == '-999':
                for i in range(len(cut_masknp)):
                    if cut_masknp[i]:
                        #count+=1
                        idxL999 += [i]
                        trkLS += [nT[i]]


        
        

        #for idx in idxL998:
            #print("998", mcLund[idx][lamidx[idx]])
        #print(idxL998)


        #print("-------------------------------")


        
        #for idx in idxL1005:
            #print("1005", mcLund[idx][lamidx[idx]])
        #print(idxL1005)

        
        print("1005 pass:", sum(trkL1005)/len(trkL1005))
        print("Sig pass:", sum(trkLS)/len(trkLS))
        print("998 pass:", sum(trkL998)/len(trkL998))
        print("All:", sum(nT)/len(nT))
        

        h = Hist.new.Reg(20, 0, 20, name="NumTRK", label=r"Number of Tracks") \
         .StrCat([], name="SP", label="SP modes", growth=True)\
         .Weight()
        
        #h.fill(NumTRK=nT, SP="All Tracks",  weight=1/102284)
        h.fill(NumTRK=trkL998, SP="998 Pass",  weight=1/93)
        #h.fill(NumTRK=trkL1005, SP="1005 Pass",  weight=1/12)
        h.fill(NumTRK=trkLS, SP="Signal Pass",  weight=1/28192)

        h.stack('SP')[:].project("NumTRK").plot(stack=False,histtype="step")
        plt.xlabel("Number of Tracks")
        plt.ylabel('Counts')
        plt.legend()
        plt.show()

        h.stack('SP')[:].project("NumTRK").plot(stack=True,histtype="fill")
        plt.xlabel("Number of Tracks")
        plt.ylabel('Counts')
        plt.legend()
        plt.show()
        
        """plt.figure()
        plt.hist(nT, bins=10, range = (0,20))
        counts, bins = np.histogram(nT, bins=10, range=(0,20))
        print(sum(counts))
        plt.xlabel("All Num Tracks")
        plt.ylabel('Counts')
        plt.figure()
        plt.hist(trkL998, bins=10, range = (0,20))
        counts, bins = np.histogram(trkL998, bins=10, range=(0,20))
        print(sum(counts))
        plt.xlabel("998 Pass Num Tracks")
        plt.ylabel('Counts')
        plt.figure()
        plt.hist(trkL1005, bins=10, range = (0,20))
        counts, bins = np.histogram(trkL1005, bins=10, range=(0,20))
        print(sum(counts))
        plt.xlabel("1005 Pass Num Tracks")
        plt.ylabel('Counts')
        plt.figure()
        plt.hist(trkLS, bins=10, range = (0,20))
        counts, bins = np.histogram(trkLS, bins=10, range=(0,20))
        print(sum(counts))
        plt.xlabel("Signal Num Tracks")
        plt.ylabel('Counts')



        plt.legend()
        plt.show()"""


        """
        lenList1005CUT = []
        lenList1005 = []
        lenList999 = []

        dict1005 = {}
        dict1005CUT = {}
        dict999 = {}
        

        indDict = {}
        for i in range(len(idxL1005)):
            
            idx1005 = idxL1005[i]
            #print("998")
            #print(mcLund[idx998][lamidx[idx998]])
            #print(mcLund[idx998])
            for particle in mcLund[idx1005]:
                
                if particle in dict1005:
                    dict1005[particle]+=1
                else:
                    dict1005[particle]=1

            lenList1005 += [len(mcLund[idx1005])]

        print("1005 events:", len(idxL1005))
        

        for i in range(len(lenList1005CUT)):
            
            idx1005CUT = lenList1005CUT[i]
            #print("998CUT")
            #print(mcLund[idx998CUT][lamidx[idx998CUT]])
            #print(mcLund[idx998CUT])
            #print(len(mcLund[idx998CUT]))
            lenList1005CUT += [len(mcLund[idx1005CUT])]
            for particle in mcLund[idx1005CUT]:
                
                if particle in dict1005CUT:
                    dict1005CUT[particle]+=1
                else:
                    dict1005CUT[particle]=1

        print("1005CUT events:", len(idxL1005CUT))

        
        for i in range(len(idxL999)):
            
            idx999 = idxL999[i]
            #print("-999")
            #print(mcLund[idx999][lamidx[idx999]])
            #print(mcLund[idx999])
            #print(len(mcLund[idx999]))
            lenList999 += [len(mcLund[idx999])]
            for particle in mcLund[idx999]:
                if particle in dict999:
                    dict999[particle]+=1
                else:
                    dict999[particle]=1

        print("-999 events:", len(idxL999))
        

        C1005 = 0
        C1005CUT = 0
        C999 = 0

        for key, value in dict1005.items():
            C1005 += value
        for key, value in dict1005CUT.items():
            C1005CUT += value
        for key, value in dict999.items():
            C999 += value

        dict1005 = dict(sorted(dict1005.items(), key=lambda item: item[1], reverse=True))
        print("Proportion of total particles by particle in 998 passing, 998 Cut, and 999 passing")
        for key, value in dict1005.items():
            if key in dict1005CUT and key in dict999:
                print(f"Particle: {key} / Count in 1005 passing: {value} / 1005 passing all prop: {value/C1005} / 1005 not passing prop: {dict1005CUT[key]/C1005CUT} / 999 passing all prop: {dict999[key]/C999}")
            elif key in dict999:
                print(f"Particle: {key} / Count in 1005 passing: {value} / 1005 passing all prop: {value/C1005} / 999 passing all prop: {dict999[key]/C999}")
            elif key in dict1005CUT:
                print(f"Particle: {key} / Count in 1005 passing: {value} / 1005 passing all prop: {value/C1005} / 1005 not passing prop: {dict1005CUT[key]/C1005CUT}")
            else:
                print(f"Particle: {key} / Count in 1005 passing: {value} / 1005 passing all prop: {value/C1005}")
        


        """
                

                       